# 리포트 06 — 실측 — 검출 사슬과 파형 순위를 결판내는 세션을 설계했다

> ### 한 일
> **보유 장비 USRP X410 로 검출 사슬과 세 파형의 상대순위를 결판내는 세션을 설계하고, 시뮬 주장마다 그것을 결정하는 측정과 판정 기준을 수치로 고정했다.**

### 결과
1. 판정 대상은 파형 순위다 — 자세평균 σ 에서 기체 5 종 ⟨outputs/report06_derived.json : ranking_validation.n_drones⟩이 같은 순위에 합의하고, 순위를 뒤집는 폭은 최소 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 다.
2. 세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ 가 그 폭 아래에 있다 — 여유 +0.30 dB ⟨outputs/report06_derived.json : ranking_validation.drift_margin_db⟩. 설계가 판정 대상과 맞는다.
3. 원거리장은 최대 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩, 점표적 서브밴드는 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩, 방위 표본은 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ 다 — 기체 2종 × 3밴드 전부를 한 거리·한 규약으로 덮는다.
4. 교정 기준체는 반경 17.8 cm ⟨outputs/report06_derived.json : calibration_pick.radius_cm⟩ 정밀 PEC 구다 — 두 기체·세 밴드에서 예상 σ 보다 최소 +3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩ 밝고, 이 구가 지금 우리 PO 출력인 **절대 레벨을 측정에 앵커한다**(생산 모드의 평균 레벨이동 0.00 dB ⟨outputs/report06_derived.json : modes.level_shift_production_abs_max_db⟩).
5. 기울기 판정 문턱은 세션간 진폭 재현성 2.44 dB ⟨outputs/report06_derived.json : slope.gap_db_min⟩, 크기법칙 판정은 두 기체의 차등신호 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 의 부호다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 판정 대상 | 자세평균 σ 위의 파형 순위 뒤집힘 폭을 캠페인 요구치로 삼음 — `benchmark/sigma_sensitivity.py:470` |
| 3층 설계 | 1층 σ(f) 레인지(§2) · 2층 ISM 5.8 GHz ⟨outputs/report06_derived.json : layers.carrier_ism_ghz⟩ 한 반송파의 파형축 · 3층 비행 검출로 나눈다 — 층마다 여는 축이 다르고 §2-6 이 그 분업과 대가를 적는다 |
| 수신 채널 | 검증 3점은 수신전용이고 기준 1 + 감시 1 = 2 채널 ⟨outputs/report06_derived.json : layers.n_channels⟩ 을 같은 클럭에서 쓴다 — 하드웨어 사양의 4 RX 와 구별한다 |
| 원거리장 요구거리 | 메쉬 외접상자(회전 로터 디스크 포함)의 3D 대각 D 를 세 밴드 λ 에 넣어 2D²/λ 로 계산 — `benchmark/plan_measurement.py` |
| 교정구 기준 σ | 정확 Mie 급수로 계산 — `benchmark/mie_pec_sphere.py:207`, `selfcheck()` 보유 |
| 기체 예상 σ | 설계 계산용 예측 — Das 앵커 레벨에 크기법칙 L² · L⁴ 를 둘 다 적용 (`src/sigma_anchor.py:255`). 생산 σ 원장은 기울기만 앵커한 `slope_only` 다(§3-2) |
| 판정 기준 | 시뮬의 주장마다 그것을 뒤집는 관측량을 짝지어 임계를 수치로 고정 — §4 결정표 |
| 하드웨어 사양 | ni.com / ettus.com 공식 스펙 한 곳에서 인용 — `src/experiment_x410.py:61` |

### 재현

```bash
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/plan_measurement.py
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python -c "import sigma_anchor as S; S.write_measurement_plan()"
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python src/make_report06_measurement.py
```

| | |
|---|---|
| 출력 | `outputs/report06_measurement.json`, `outputs/measurement_plan.json`, `outputs/report06_derived.json` |
| 소요 | 약 10 초 (CPU) |
| 비고 | 산문판 설계서는 `docs/MEASUREMENT_PLAN.md` 이고, 그 안의 수치표는 `src/sigma_anchor.py:939` 가 자동 주입한다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 02 §4 | 앵커가 통제한 항과 통제되지 않은 항의 크기 원장 |
| 03 | 세 조명원(LTE · 5G · WiFi)의 대역과 점유 |
| 04 | 명목 Pfa 를 경험 Pfa 로 교정하는 절차 |
| 05 | 자유공간 탐지 결과가 서 있는 기하 |

<!--pk:paper_map {"kind": "paper_map", "sections": ["VI. Validation"], "claim": "X410 야외 캠페인은 검출 사슬과 세 파형의 상대순위를 결판내며, 그 판정에 필요한 진폭 재현성을 세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ 로 확보한다 — 자세평균 순위를 뒤집는 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 아래다.", "evidence": ["§4 결정표", "§3", "그림 5", "그림 6", "outputs/report06_derived.json:ranking_validation.flip_span_min_db", "outputs/report06_derived.json:farfield_adopted.R_ff_max_m", "outputs/sigma_sensitivity.json:aspect_averaged.consensus_order", "outputs/verify_cfar.json:meta.runtime_s"], "qualifications": ["절대 탐지거리와 Pfa 교정은 §4 표에서 '이 캠페인 밖' 으로 표시한다 — 환경 공통항과 통제 몬테카를로가 각각 정한다", "σ 절대레벨은 교정구가 세션 안에서 앵커한다 (§2-2). 기울기 앵커는 Das 측정이다"], "report": "report06_measurement"}-->
> **논문 대응** · **VI. Validation**
>
> 주장 — X410 야외 캠페인은 검출 사슬과 세 파형의 상대순위를 결판내며, 그 판정에 필요한 진폭 재현성을 세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ 로 확보한다 — 자세평균 순위를 뒤집는 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 아래다.
> 근거 — §4 결정표 · §3 · 그림 5 · 그림 6 · `outputs/report06_derived.json:ranking_validation.flip_span_min_db` · `outputs/report06_derived.json:farfield_adopted.R_ff_max_m` · `outputs/sigma_sensitivity.json:aspect_averaged.consensus_order` · `outputs/verify_cfar.json:meta.runtime_s`
> 단서 — 절대 탐지거리와 Pfa 교정은 §4 표에서 '이 캠페인 밖' 으로 표시한다 — 환경 공통항과 통제 몬테카를로가 각각 정한다 · σ 절대레벨은 교정구가 세션 안에서 앵커한다 (§2-2). 기울기 앵커는 Das 측정이다

---

## §1. 하드웨어 — X410 한 대가 기준과 감시를 동시에 든다

세션은 **RX0 = 기준(직접파)** 과 **RX1 = 감시** 두 채널을 **같은 클럭**에서 쓴다⟨outputs/measurement_layers.json : validation_three_points.channels⟩ — 사양의 4 RX 는 각도축을 여는 예비다.
사양은 `src/experiment_x410.py:61`, 기하 배치는 `src/experiment_x410.py:100` 한 곳에 있다.

12-bit ADC 의 동적범위 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ 가 직접파 제거의 천장이고,
직접파를 양자화한 뒤 남는 잔차를 `src/experiment_x410.py:83` 의 `adc_quantize()` 가 모델에 넣는다.

| 항목 | 값 | 무엇을 제약하나 |
|---|---|---|
| TX / RX 채널 | 4 ⟨outputs/report06_measurement.json : hw.n_tx⟩ / 4 ⟨outputs/report06_measurement.json : hw.n_rx⟩ | 세션은 기준 1 + 감시 1 = 2 채널 ⟨outputs/report06_derived.json : layers.n_channels⟩ 을 공통 클럭에서 쓴다 |
| 채널당 순시대역 | 400 MHz ⟨outputs/report06_measurement.json : hw.max_bw_mhz⟩ | 거리분해능과 점표적 서브밴드(§2-3) |
| 주파수 범위 | 1 MHz ⟨outputs/report06_derived.json : hw_span.f_lo_mhz⟩ ~ 7.2 GHz ⟨outputs/report06_derived.json : hw_span.f_hi_ghz⟩ | 세 밴드 전부 커버 |
| ADC 동적범위 | 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ | 직접파 제거의 천장 |
| 감시배열 AoA 빔폭 | 33.8° ⟨outputs/report06_measurement.json : hw.aoa_beamwidth_deg⟩ | 네 RX 를 전부 감시로 쓸 때 열리는 각도축 — 디텍션 이후의 확장축 |
| 최대대역 바이스태틱 ΔR | 0.749 m ⟨outputs/report06_measurement.json : hw.range_res_bistatic_m_at_max_bw⟩ | 표적이 퍼지는 폭(§2-3) |

원사양 출처는 `src/experiment_x410.py:61 (ni.com / ettus.com 2024 spec)` 한 곳이다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_adc_headroom.png", "figure_no": "1", "question": "12-bit ADC 는 직접파 대 잡음비 위에 얼마의 여유를 남기는가?", "paper_caption": "Headroom of the 12-bit ADC dynamic range above the direct-path-to-noise ratio for the three waveforms, tightest for the LTE waveform.", "vector_pdf": "outputs/figures/report06_adc_headroom.pdf", "report": "report06_measurement"}-->
![report06_adc_headroom.png](outputs/figures/report06_adc_headroom.png)

**그림 1.** 12-bit ADC 는 직접파 대 잡음비 위에 얼마의 여유를 남기는가?

여유가 가장 좁은 파형은 `LTE20` 이고 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ 다 — 점유대역이 좁아 기준채널 이득이 높다.

이 DNR 은 자유공간 시뮬 기하에서 나온 값이다⟨outputs/report06_measurement.json : adc.dnr_source⟩. 야외에서 송수신을 가깝게 놓으면 DNR 이 올라가 여유가 그만큼 줄어든다.

## §2. ⭐ 교정된 절대 σ 로 가는 세션 — 실행 체크리스트 6항목

이 여섯 항목이 **교정된 절대 σ** 를 만드는 조건 전부다 — 기준체 · 배경차감 · 자세통제 · 원거리장 · 점표적 대역 · 패턴교정. 순위 판정(§3)은 이 중 앞의 셋만 요구하고, 여섯을 다 채운 세션이 다음 라운드의 더 센 주장(절대 σ)을 만든다.

왼쪽은 세션에서 **하는 일**, 오른쪽은 그 일이 만족해야 하는 **수치 임계**다.

| 실행 항목 | 세션에서 하는 일 | 수치 임계 |
|---|---|---|
| 교정 기준체 | 정밀 PEC 구를 세션 **시작과 끝**에 표적과 같은 지지대·같은 위치에서 잰다 | 예상 σ 대비 여유 ≥ +3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩ (§2-2) |
| 배경 차감 | 지지대를 세운 채 표적만 치우고 배경 응답을 **복소수로** 뺀다 | 지면반사 경로차 > ΔR — 기하 78% ⟨outputs/report06_derived.json : ground_bounce_sep_frac_200MHz⟩ 가 분리 (§2-5) |
| 자세 통제 | 엔코더 턴테이블로 방위를 돌리고 로터를 정지시켜 블레이드 방위를 기록한다 | Δφ ≤ 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ (§2-4) |
| 안테나 패턴 교정 | 교정구를 표적과 같은 자리·같은 높이에 놓아 패턴과 체인 이득을 비율로 소거한다 | 표적/교정구 위치 동일 (§2-2) |
| 원거리장 거리 | 2D²/λ 이상에서 잰다 — 교정구는 같은 자리에 놓으면 자동 만족한다 | R ≥ 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ (§2-1) |
| 점표적 서브밴드 | 400 MHz ⟨outputs/report06_measurement.json : hw.max_bw_mhz⟩ 순시대역을 쪼개 서브밴드마다 σ 를 내고, 그 다발을 σ(f) 로 삼는다 | B ≤ 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩ (§2-3) |

여섯 항목의 산문판 조건은 `docs/MEASUREMENT_PLAN.md` §1-1~1-6 에 있다.

### §2-1. 원거리장 — 2D²/λ 를 두 기체 × 세 밴드로 계산했다

채택한 D 는 가장 보수적인 정의다 — 회전 로터 디스크까지 포함한 외접상자의 3D 대각.
같은 기체를 모터-모터 대각으로 재정의하면 요구거리가 3.65 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩배 짧아진다. 세 정의를 한 표에 나란히 실어 어느 값을 쓰는지 고정한다.

| 기체 | 밴드 | λ | D_env | **R_ff(env)** | R_ff(bbox) | R_ff(모터대각) |
|---|---|---|---|---|---|---|
| matrice4e | LTE | 163 mm | 0.839 m | 8.65 m | 4.42 m | 2.37 m |
| matrice4e | 5G | 86 mm | 0.839 m | 16.42 m | 8.38 m | 4.50 m |
| matrice4e | WiFi | 58 mm | 0.839 m | 24.44 m | 12.48 m | 6.69 m |
| mini5pro | LTE | 163 mm | 0.497 m | 3.03 m | 1.74 m | 0.93 m |
| mini5pro | 5G | 86 mm | 0.497 m | 5.76 m | 3.30 m | 1.77 m |
| mini5pro | WiFi | 58 mm | 0.497 m | 8.57 m | 4.91 m | 2.63 m |

출처 ⟨outputs/report06_derived.json : farfield⟩

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_farfield.png", "figure_no": "2", "question": "각 기체와 밴드에서 원거리장에 들어가려면 얼마나 멀어야 하는가?", "paper_caption": "Far-field distance 2D^2/lambda for the two purchased airframes at the three bands, under three definitions of the aperture D.", "vector_pdf": "outputs/figures/report06_farfield.pdf", "report": "report06_measurement"}-->
![report06_farfield.png](outputs/figures/report06_farfield.png)

**그림 2.** 각 기체와 밴드에서 원거리장에 들어가려면 얼마나 멀어야 하는가?

세션 거리는 최대값 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ 로 잡는다 — 그 한 거리가 두 기체 세 밴드를 전부 덮는다.

### §2-2. 교정 기준체 — σ 를 절대량으로 만드는 장치

**정밀 PEC 구**를 쓴다. 구는 방위무관이라 정렬 오차가 σ 에 안 들어간다.

기준값은 **정확 Mie** 로 쓴다 — πr² 광학 점근과의 차이는 아래 표의 `Mie−πr²` 열에 dB 로 있다.
단일 출처는 `benchmark/mie_pec_sphere.py:207` 이고 자체검증 `selfcheck()` 을 갖고 있다.

세션 **시작과 끝에 한 번씩** 잰다. 두 값의 차가 그 세션의 드리프트 예산이고,
그 차가 목표 정확도(≤ 1 dB) 안에 들어온 세션만 자료로 쓴다.

앵커 사슬은 같은 반경의 금속구를 σ_cal -10.00 dBsm ⟨outputs/report06_derived.json : layers.cal_anchor_declared_dbsm⟩ 로 선언했다 — πr² 광학값 -10.02 dBsm ⟨outputs/report06_derived.json : layers.cal_pir2_dbsm⟩ 이다⟨outputs/measurement_layers.json : calibration_convention_gap.anchor_quote⟩. 우리가 정확 Mie 로 교정하면 우리 σ 는 그 규약 대비 밴드에 따라 +0.07 ⟨outputs/report06_derived.json : layers.mie_shift_min_db⟩ ~ +0.31 dB ⟨outputs/report06_derived.json : layers.mie_shift_max_db⟩ 위로 뜬다 — 앵커와 사과-대-사과로 견줄 때 이 항을 먼저 되돌린다.

| 구 | 밴드 | ka | σ_Mie | Mie−πr² | Matrice 4E 대비 여유 | Mini 5 Pro 대비 여유 |
|---|---|---|---|---|---|---|
| r=17.8 cm | LTE | 6.9 | -9.95 dBsm | +0.07 dB | +4.38 dB | +8.44 dB |
| r=17.8 cm | 5G | 13.1 | -9.71 dBsm | +0.31 dB | +4.27 dB | +8.33 dB |
| r=17.8 cm | WiFi | 19.4 | -9.89 dBsm | +0.13 dB | +3.73 dB | +7.79 dB |
| r=25.0 cm | LTE | 9.7 | -6.47 dBsm | +0.60 dB | +7.86 dB | +11.92 dB |
| r=25.0 cm | 5G | 18.3 | -7.06 dBsm | +0.01 dB | +6.93 dB | +10.98 dB |
| r=25.0 cm | WiFi | 27.3 | -7.14 dBsm | -0.07 dB | +6.49 dB | +10.55 dB |

출처 ⟨outputs/report06_derived.json : calibration⟩

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_calibration.png", "figure_no": "3", "question": "어느 반경의 교정구가 세 밴드 모두에서 기체 예상 σ 위에 있는가?", "paper_caption": "Exact Mie cross section of candidate calibration spheres against the expected drone level at the three bands.", "vector_pdf": "outputs/figures/report06_calibration.pdf", "report": "report06_measurement"}-->
![report06_calibration.png](outputs/figures/report06_calibration.png)

**그림 3.** 어느 반경의 교정구가 세 밴드 모두에서 기체 예상 σ 위에 있는가?

채택 반경은 17.8 cm ⟨outputs/report06_derived.json : calibration_pick.radius_cm⟩ 다 — 세 밴드 모두 Mie−πr² 편차가 작고 앵커 문헌(Yuan)이 쓴 것과 같은 크기다.

교정구가 기체보다 최소 +3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩ 밝으므로 같은 이득 설정으로 둘 다 잡히고, 그래야 두 응답의 비율이 그대로 σ 비율이 된다.

### §2-3. 점표적 서브밴드 — 순시대역을 쪼개 σ(f) 로 만든다

peak |s|² 를 σ 로 쓰려면 표적이 **한 거리빈 안**에 들어와야 한다(ΔR = c/2B > D_bbox).
두 기체를 함께 만족시키는 최대 서브밴드는 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩ 다. 서브밴드마다 σ 를 내면 그 다발이 곧 σ(f) 이고, 앵커 문헌(Das §II-3c)의 절차와 같다.

| 대역 B | ΔR = c/2B | Matrice 4E | 여유 | Mini 5 Pro | 여유 |
|---|---|---|---|---|---|
| 400 MHz | 0.375 m | ⚠ 퍼짐 | -0.225 m | ⚠ 퍼짐 | -0.001 m |
| 200 MHz | 0.749 m | 점표적 | +0.150 m | 점표적 | +0.373 m |
| 100 MHz | 1.499 m | 점표적 | +0.900 m | 점표적 | +1.123 m |
| 50 MHz | 2.998 m | 점표적 | +2.399 m | 점표적 | +2.622 m |

출처 ⟨outputs/report06_derived.json : point_target⟩

게이팅은 넓게, 평가는 좁게 한다 — 앵커는 6차 Kaiser 창으로 CIR 을 게이팅한 뒤 주파수축으로 되돌리고⟨outputs/measurement_layers.json : gate_wide_evaluate_narrow.anchor_quote⟩, 우리는 400 MHz ⟨outputs/report06_derived.json : layers.gate_bw_mhz⟩ 전대역에서 게이팅한 뒤 σ 는 50 MHz ⟨outputs/report06_derived.json : layers.eval_bw_mhz⟩ 서브밴드 8 ⟨outputs/report06_derived.json : layers.n_subbands⟩ 개로 평가한다.

### §2-4. 자세 통제 — 각도표본을 λ/4D 로 잡았다

방위는 엔코더 턴테이블로 돌리고, 표본 간격은 `λ/4D` 이하로 잡는다.
가장 촘촘한 요구는 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ (`matrice4e` · `WiFi`)이고, 한 바퀴에 262 ⟨outputs/report06_derived.json : aspect_n_az_max⟩ 표본이다.

앵커 문헌은 밴드와 무관하게 2.00° ⟨outputs/report06_derived.json : layers.anchor_step_deg⟩ 고정(반원 91 점 ⟨outputs/report06_derived.json : layers.anchor_N⟩)을 썼고, 우리는 밴드마다 λ/4D 를 따라간다 — 요구 표본수는 반원당 28 ⟨outputs/report06_derived.json : layers.N_required_min⟩ ~ 145 ⟨outputs/report06_derived.json : layers.N_required_max⟩ 점이다.
표의 마지막 열이 밴드별 대소를 그대로 싣는다.

앵커는 높은 주파수에서 그 고정 간격이 성기다고 스스로 적었고⟨outputs/measurement_layers.json : angular_sampling._rule⟩, 우리 기체에서 그 자리는 `matrice4e @ WiFi 5.21 GHz` 와 `matrice4e @ ISM 5.8 GHz` 두 칸이다. 나머지 칸에서는 2° 가 우리 요구보다 촘촘하다.

로터는 **정지**시키고 블레이드 방위를 기록한다 — 앵커가 회전 성분을 뺐으므로 그 규약에 맞춘다.

| 기체 | 밴드 | Δφ 나이퀴스트 | Δφ 권장 | 한 바퀴 표본수 | 앵커 고정 2° 보다 촘촘한가 |
|---|---|---|---|---|---|
| matrice4e | LTE | 7.78° | 3.89° | 93 | 아니오 |
| matrice4e | 5G | 4.09° | 2.05° | 176 | 아니오 |
| matrice4e | WiFi | 2.75° | 1.38° | 262 | 예 |
| mini5pro | LTE | 12.39° | 6.20° | 58 | 아니오 |
| mini5pro | 5G | 6.53° | 3.26° | 110 | 아니오 |
| mini5pro | WiFi | 4.38° | 2.19° | 164 | 아니오 |

출처 ⟨outputs/report06_derived.json : aspect⟩

### §2-5. 배경 차감과 지면반사 — 야외 부지를 기하로 다룬다

배경 S_BG 는 **지지대를 세운 채로** 재고 **복소수로** 뺀다.
표적을 경유한 지면반사는 경로차 `2hH/R` 이 서브밴드 거리분해능보다 **클 때** 레인지게이팅으로 떨어진다.

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_ground_bounce.png", "figure_no": "4", "question": "어떤 야외 기하가 지면반사 유령을 표적 거리빈 밖으로 밀어내는가?", "paper_caption": "Target-via-ground path difference against range resolution, over antenna heights, target heights and ground ranges.", "vector_pdf": "outputs/figures/report06_ground_bounce.pdf", "report": "report06_measurement"}-->
![report06_ground_bounce.png](outputs/figures/report06_ground_bounce.png)

**그림 4.** 어떤 야외 기하가 지면반사 유령을 표적 거리빈 밖으로 밀어내는가?

27 ⟨outputs/report06_derived.json : ground_bounce_n_geom⟩개 기하 중 200 MHz ⟨outputs/report06_derived.json : ground_bounce_ref_bw_MHz⟩ 서브밴드에서 분리되는 비율은 78% ⟨outputs/report06_derived.json : ground_bounce_sep_frac_200MHz⟩ 다. 부지 선정은 그림 4 에서 분해능 선 위에 오는 (h, H, R) 조합으로 한다.

### §2-6. 세 층 — §2 는 1층이고, 파형축과 비행검출이 그 위에 선다

| 층 | 무엇을 재나 | 반송파 | 산출 |
|---|---|---|---|
| 1층 — σ(f) 레인지 | 정지 표적 · 턴테이블 방위컷 · 교정구 · 배경 코히런트 차감 | LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz (§2) | σ(f, φ) 의 분포 P(σ) |
| 2층 — 파형축 | 세 파형 구조를 같은 서브밴드 중심에 겹쳐 송신 | ISM 5.8 GHz 단일 (span 150 MHz) | SNR_out / E_tx |
| 3층 — 비행 검출 | 로터가 도는 비행 표적 — 로터가 도는 유일한 층 | ISM 5.8 GHz | 고정 Pfa 에서 Pd(range) |

출처 ⟨outputs/report06_derived.json : layers.rows⟩

2층을 한 반송파에 고정하는 이유는 면허다 — 2.1 GHz 야외 송신은 허가가 필요하고 '2.1 GHz 의 WiFi' 라는 배치신호는 세상에 존재하지 않는 인공물이다⟨outputs/measurement_layers.json : layer2_waveform_axis.why_one_carrier⟩. 잃는 반송파축은 수신전용 검증 3점(LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz ⟨outputs/report06_derived.json : layers.validation_points⟩)의 실제 배치신호와 v_max 의 λ 비 이전으로 갚는다 — 그 3점은 교차설계의 대각선이 아니라 독립 검사점이다.

1층이 내는 것은 점별 패턴이 아니라 **분포 P(σ)** 다 — 검출확률이 σ 분포의 함수이므로, 그 분포를 Swerling 틀에 넣어 3층의 `Pd vs range at fixed Pfa` 를 예측하고 3층이 그 예측을 검사한다⟨outputs/measurement_layers.json : layer3_flight.ties_back_to⟩. 2층·3층의 ISM 원거리장은 bbox 정의로 최대 13.69 m ⟨outputs/report06_derived.json : layers.farfield_ism_bbox_max_m⟩ 이고, §2-1 이 채택한 세션 거리 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩(env 정의) 안에 든다.

## §3. ⭐ 시뮬과 실측 — 캠페인이 결판내는 양은 순위다

자세평균 σ 에서 기체 5 종 ⟨outputs/report06_derived.json : ranking_validation.n_drones⟩이 순위 `L1 > G1 > W1` 에 합의하고, 그 순위를 뒤집는 밴드별 σ 이동폭은 Matrice 4E 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_db.matrice4e⟩ · Mini 5 Pro 2.95 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_db.mini5pro⟩ 다.
세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩(§2-2)가 그 폭 아래에 있으므로(여유 +0.30 dB ⟨outputs/report06_derived.json : ranking_validation.drift_margin_db⟩) 이 캠페인은 **검출 사슬과 파형 순위**를 결판낸다.
절대 σ 는 §2 여섯 항목을 다 채운 세션이 다음 라운드에서 만든다.

| 설계상 같게 맞춘 축 | 설계상 다르게 둔 축 |
|---|---|
| 바이스태틱 구조 — 기준 1 + 감시 1, 공통 클럭 | 환경 — 시뮬은 자유공간, 실측은 지면반사·다중경로 |
| 같은 기체 2종 (Matrice 4E · Mini 5 Pro) | 클러터 — 실측 부지의 정적 산란체 |
| 같은 세 파형 (LTE · 5G · WiFi) | 동적범위 — 시뮬 ECA 는 float64, 실측은 12-bit |
| 같은 검출기 사슬 (ECA → 거리도플러 → CA-CFAR) | 자세 — 시뮬은 각도격자, 비행 중에는 자유 |
| σ 통계 규약 (방위 선형평균) | 링크예산 전제 — 절대 탐지거리 |

### §3-1. 앵커 원장 — 이 캠페인이 닫으러 가는 항목과 그 크기

| 항목 | 상태 | 크기 |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 dB |
| size transfer law | UNRESOLVED | +9.50 dB |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 dB |
| near-field vs far-field, environment | OK | +0.00 dB |

출처 ⟨outputs/sigma_anchor.json : uncontrolled⟩

### §3-2. 재보정 모드 — 이 편의 어느 숫자가 어느 모드에서 오나

생산 σ 원장은 `slope_only ⟨outputs/report06_derived.json : modes.production_mode⟩` 다 — 주파수 기울기만 측정에서 받고 **절대 레벨은 우리 PO 출력 그대로**다(평균 레벨이동 0.00 dB ⟨outputs/report06_derived.json : modes.level_shift_production_abs_max_db⟩). 레벨을 앵커에 맞추는 두 모드는 이 편의 설계 계산에만 쓴다.

| 모드 | 무엇을 옮기나 | Matrice 4E | Mini 5 Pro | 이 편에서 쓰는 곳 |
|---|---|---|---|---|
| slope_only (생산 기본) | 주파수 기울기만 | +0.00 dB | +0.00 dB | 생산 σ 원장 (02편 §4) · §4 결정표 |
| level_and_slope_L2 | 레벨 + 기울기 (L²) | +6.47 dB | +4.03 dB | §2-2 예상 σ — 교정구 반경·동적범위 설계 |
| level_and_slope_L4 | 레벨 + 기울기 (L⁴) | +8.44 dB | +1.93 dB | §4-2 크기법칙 차등신호 |

출처 ⟨outputs/report06_derived.json : modes.rows⟩

교정구를 표적과 같은 자리에서 재는 §2-2 가 이 표의 첫 줄을 바꾼다 — 레벨이 우리 계산에서 우리 측정으로 옮겨 온다.

| 기체 | 앵커 대비 등급 | 크기비 | L² 보정 | L⁴ 보정 |
|---|---|---|---|---|
| Matrice 4E | scaled | 1.254 ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_ratio⟩ | +1.96 dB ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_corr_L2_db⟩ | +3.93 dB ⟨outputs/sigma_anchor.json : drones.matrice4e.comparability.size_corr_L4_db⟩ |
| Mini 5 Pro | scaled | 0.786 ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_ratio⟩ | -2.09 dB ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_corr_L2_db⟩ | -4.19 dB ⟨outputs/sigma_anchor.json : drones.mini5pro.comparability.size_corr_L4_db⟩ |

두 기체 다 등급이 `scaled` 다 — 앵커 기체와 같은 4로터 위상이고 대각만 다르다.
이 원장의 각 행이 §4 결정표 왼쪽 열과 1:1 로 붙는다.

## §4. ⭐ 결정표 — 어느 측정이 어느 주장을 결판내는가

네 번째 열이 **판정 범위**다 — `결판`(이 캠페인이 정한다) · `사슬 확인`(설계값과 대조한다) · `이 캠페인 밖`(다음 라운드나 통제 시뮬이 정한다). 세 번째 값을 그대로 적는 것이 이 표의 핵심이다.

| 02편의 주장 | 이를 결정하는 측정 | 판정 기준 | 판정 범위 |
|---|---|---|---|
| 자세 패턴 B1(φ) 을 SBR+PO 기하에서 계산했다 | 턴테이블 방위컷, 로터 정지 | Δφ ≤ 1.38° ⟨outputs/report06_derived.json : aspect_finest_deg⟩ 로 재고 로브 위치 대조 (§2-4) | 결판 |
| 절대 레벨은 우리 PO 출력이다 (앵커는 기울기만 옮긴다) | 표적과 같은 자리에서 교정구 + 배경 코히런트 차감 — **레벨의 첫 측정 앵커** | 교정구 여유 ≥ +3.73 dB ⟨outputs/report06_derived.json : calibration_margin_min_db⟩, 세션 드리프트 ≤ 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ (§2-2). 눈감기 사전값 — 고도정합 실측곡선 대비 밴드평균 -4.91 dB ⟨outputs/p3_validation.json : residual.vs_yuan_theta90_measured_curve.mean_db⟩ | 결판 |
| 모서리 회절 — 1차 PTD 항이 커널에 있고 생산 기본값은 끔이다 (02) | 모서리가 많은 표준체(평판·이면각)를 같은 세션에서 함께 측정 | 위상까지 정합해 부호를 심판한다 — 평판 RMS 시험은 위상맹목이다. 켠 비용은 +47.2% ⟨outputs/ptd_wiring.json : verdict.cost_increase_pct⟩ (§5) | 결판 |
| 밴드 기울기는 Das 의 0.210 ⟨outputs/report06_derived.json : slope.anchor_db_per_ghz⟩ dB/GHz 로 맞췄다 | 세 밴드를 같은 세션에서 측정 | 세션 재현성 < 2.44 dB ⟨outputs/report06_derived.json : slope.gap_db_min⟩ (§4-1) | 결판 |
| 크기전이는 L² 와 L⁴ 를 괄호로 함께 싣는다 (9.50 dB ⟨outputs/report06_derived.json : size_law.uncontrolled_size_db⟩) | 두 기체를 한 캠페인에서 측정 | 차등 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 의 부호 (§4-2) | 결판 |
| 편파: VV 단일, 커널은 무편파 스칼라 | VV / VH / HV / HH 4조합 | 무편파 모형과 VV 측정의 차를 dB 로 확정 | 결판 |

| 03~05편의 주장 | 이를 결정하는 측정 | 판정 기준 | 판정 범위 |
|---|---|---|---|
| 파형별 점유·대역폭 대가는 σ 와 무관하게 정확하다 (03) | X410 이 같은 기하에서 세 파형을 송신 | 측정 ΔR·모호함수가 설계값과 일치 | 사슬 확인 |
| 파형 상대순위 `L1 > G1 > W1` (05) | 야외 고정기하 탐지시험 · 방위 스윕으로 자세평균 | 순서 일치 · 뒤집힘 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ (§3) | 결판 |
| 자유공간 절대 탐지거리 (05) | 환경 공통항(지면·클러터)이 정한다 | σ 공통이동 1 dB 당 거리 0.25 dB ⟨outputs/report06_derived.json : ranking_validation.common_mode_slope_db_per_db⟩ 이동 | **이 캠페인 밖** |
| 명목 Pfa 를 교정해야 파형 비교가 성립한다 (04) | 부지 배경 CPI 를 녹화해 그 부지의 경험 Pfa 를 별도로 기록 | 교정 자체는 통제 몬테카를로 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ 가 세운다 | **이 캠페인 밖** |
| 12-bit ADC 가 직접파 제거의 천장이다 (§1) | 직접파를 실제로 받아 ECA 잔차를 잰다 | 여유 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ 가 야외에 얼마나 남나 | 사슬 확인 |

### §4-1. 기울기 — 세션 재현성이 판정의 문턱이다

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_slope.png", "figure_no": "5", "question": "우리 기울기와 앵커 기울기를 가르려면 세션 재현성이 얼마나 좋아야 하는가?", "paper_caption": "Frequency-slope hypotheses across the measured band span and the gap they open at the top of that span.", "vector_pdf": "outputs/figures/report06_slope.pdf", "report": "report06_measurement"}-->
![report06_slope.png](outputs/figures/report06_slope.png)

**그림 5.** 우리 기울기와 앵커 기울기를 가르려면 세션 재현성이 얼마나 좋아야 하는가?

우리 커널은 0.936 ⟨outputs/report06_derived.json : slope.rows[0].ours_db_per_ghz⟩ ~ 1.517 ⟨outputs/report06_derived.json : slope.rows[1].ours_db_per_ghz⟩ dB/GHz 이고 앵커는 0.210 ⟨outputs/report06_derived.json : slope.anchor_db_per_ghz⟩ dB/GHz 다. 대역 3.367 GHz ⟨outputs/report06_derived.json : slope.span_ghz⟩ 를 지나며 두 가설이 2.44 ⟨outputs/report06_derived.json : slope.gap_db_min⟩ ~ 4.40 ⟨outputs/report06_derived.json : slope.rows[1].gap_db⟩ dB 벌어진다. 기울기의 정의는 하나로 고정한다 — 세 밴드 방위평균 μ 를 f[GHz] 에 1차 적합(el=0) ⟨outputs/report06_derived.json : slope.fit_note_short⟩.

### §4-2. 크기법칙 — 두 기체를 함께 사는 이유

<!--pk:figure {"kind": "figure", "path": "outputs/figures/report06_size_law.png", "figure_no": "6", "question": "두 기체를 함께 재면 L² 와 L⁴ 를 가를 수 있는가?", "paper_caption": "Mean cross section predicted by the L^2 and L^4 size-transfer laws for one airframe larger and one smaller than the measurement anchor.", "vector_pdf": "outputs/figures/report06_size_law.pdf", "report": "report06_measurement"}-->
![report06_size_law.png](outputs/figures/report06_size_law.png)

**그림 6.** 두 기체를 함께 재면 L² 와 L⁴ 를 가를 수 있는가?

Matrice 4E 는 앵커보다 크고(크기비 1.254 ⟨outputs/report06_derived.json : size_law.by_airframe.matrice4e.size_ratio⟩), Mini 5 Pro 는 작다(0.786 ⟨outputs/report06_derived.json : size_law.by_airframe.mini5pro.size_ratio⟩). 두 법칙의 예측이 **반대 방향**으로 갈리므로 두 기체의 μ 차이가 법칙을 직접 고른다.

차등신호 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 가 두 기체를 함께 사는 이유다.

<!--pk:methods {"kind": "methods", "report": "report06_measurement", "text": "Validation campaign design. Two airframes (DJI Matrice 4E, DJI Mini 5 Pro) are measured with a USRP X410 (4 TX / 4 RX, 400 MHz ⟨outputs/report06_measurement.json : hw.max_bw_mhz⟩ instantaneous bandwidth per channel, 12-bit ADC giving 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ of dynamic range), each session receiving on two channels driven by a common clock, RX0 as the reference and RX1 as the surveillance channel. The campaign is organised in three layers: a static cross-section range that yields sigma(f) and its distribution over aspect, a waveform axis measured at a single ISM carrier of 5.8 GHz ⟨outputs/report06_derived.json : layers.carrier_ism_ghz⟩ because outdoor transmission at the cellular carriers is licensed, and a flight-detection layer that reads Pd against range at a fixed CFAR design Pfa; the carrier axis given up in the second layer is recovered by receiving the three deployed signals (LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz) and by transferring the unambiguous velocity with the wavelength ratio. The session range is 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩, the largest 2D^2/lambda over both airframes and all three bands with D taken as the enclosing-box diagonal including the rotor discs. Absolute level is tied to a precision PEC sphere of radius 17.8 cm ⟨outputs/report06_derived.json : calibration_pick.radius_cm⟩ measured at the target position at the start and the end of every session against an exact Mie reference, with a session drift budget of 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩; the background is recorded with the mount in place and subtracted coherently. The capture is split into sub-bands of 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩ so that c/2B exceeds the airframe extent, and each sub-band yields one cross section, giving sigma(f). Azimuth is stepped on an encoder turntable at 1.38 deg ⟨outputs/report06_derived.json : aspect_finest_deg⟩, at or below lambda/4D in every band, with the rotors stopped and the blade azimuth logged. Detection reuses the simulation chain (ECA, range-Doppler, CA-CFAR) unchanged.", "tools": ["Python 3.12", "NumPy 2.5", "SciPy 1.18", "Sionna 2.0.1", "PyTorch 2.12"], "params": ["1.00 dB", "1.38 deg", "1.843 GHz", "17.8 cm", "200 MHz", "24.44 m", "3.5 GHz", "400 MHz", "5.21 GHz", "5.8 GHz", "74.01 dB"], "versions": ["LTE 1.84", "NumPy 2.5", "PyTorch 2.12", "Python 3.12", "SciPy 1.18", "Sionna 2.0.1", "WiFi 5.2"], "n_words": 315}-->
### §4.5 방법 문단 (논문 이관용)

Validation campaign design. Two airframes (DJI Matrice 4E, DJI Mini 5 Pro) are measured with a USRP X410 (4 TX / 4 RX, 400 MHz ⟨outputs/report06_measurement.json : hw.max_bw_mhz⟩ instantaneous bandwidth per channel, 12-bit ADC giving 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ of dynamic range), each session receiving on two channels driven by a common clock, RX0 as the reference and RX1 as the surveillance channel. The campaign is organised in three layers: a static cross-section range that yields sigma(f) and its distribution over aspect, a waveform axis measured at a single ISM carrier of 5.8 GHz ⟨outputs/report06_derived.json : layers.carrier_ism_ghz⟩ because outdoor transmission at the cellular carriers is licensed, and a flight-detection layer that reads Pd against range at a fixed CFAR design Pfa; the carrier axis given up in the second layer is recovered by receiving the three deployed signals (LTE 1.843 GHz · 5G NR 3.5 GHz · WiFi 5.21 GHz) and by transferring the unambiguous velocity with the wavelength ratio. The session range is 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩, the largest 2D^2/lambda over both airframes and all three bands with D taken as the enclosing-box diagonal including the rotor discs. Absolute level is tied to a precision PEC sphere of radius 17.8 cm ⟨outputs/report06_derived.json : calibration_pick.radius_cm⟩ measured at the target position at the start and the end of every session against an exact Mie reference, with a session drift budget of 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩; the background is recorded with the mount in place and subtracted coherently. The capture is split into sub-bands of 200 MHz ⟨outputs/report06_derived.json : point_target_max_bw_MHz⟩ so that c/2B exceeds the airframe extent, and each sub-band yields one cross section, giving sigma(f). Azimuth is stepped on an encoder turntable at 1.38 deg ⟨outputs/report06_derived.json : aspect_finest_deg⟩, at or below lambda/4D in every band, with the rotors stopped and the blade azimuth logged. Detection reuses the simulation chain (ECA, range-Doppler, CA-CFAR) unchanged.

버전 — `Python 3.12` · `NumPy 2.5` · `SciPy 1.18` · `Sionna 2.0.1` · `PyTorch 2.12`

<!--rs:paper-->
<!--pk:defence {"kind": "defence", "report": "report06_measurement", "rows": [{"주장": "야외 캠페인은 검출 사슬과 세 파형의 상대순위를 결판낸다.", "근거": "§3 · §4 결정표 · outputs/sigma_sensitivity.json:aspect_averaged.consensus_order", "공격": "야외 환경이 시뮬 자유공간과 달라 비교 대상이 흐려진다.", "답": "순위를 정하는 λ²·점유·대역폭 항은 밴드 간 차이고 환경 항은 세 밴드 공통이다. 자세평균 σ 에서 기체 5 종 ⟨outputs/report06_derived.json : ranking_validation.n_drones⟩이 같은 순위에 합의한다 ⟨outputs/sigma_sensitivity.json : aspect_averaged.all_drones_agree⟩."}, {"주장": "세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ 가 자세평균 뒤집힘 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 아래에 있다.", "근거": "§3 · outputs/report06_derived.json:ranking_validation.drift_margin_db", "공격": "여유 0.30 dB ⟨outputs/report06_derived.json : ranking_validation.drift_margin_db⟩ 는 얇다.", "답": "좁은 쪽은 Matrice 4E 하나이고 Mini 5 Pro 는 2.95 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_db.mini5pro⟩ 다. 두 기체를 함께 재어 넓은 쪽이 좁은 쪽의 판정을 받쳐 준다."}, {"주장": "교정구를 표적 자리에서 세션 시작·끝에 재어 절대 레벨의 첫 측정 앵커를 세운다.", "근거": "그림 3 · outputs/report06_derived.json:calibration_margin_min_db", "공격": "그 여유는 앵커 절대레벨 위의 설계값이고, 앵커 절편에는 통계 규약 변환 상수가 들어 있다.", "답": "그 상수는 2.51 dB ⟨outputs/sigma_anchor.json : statistic_resolution.reconcile.by_kind.exponential.offset_db⟩ 이고, 빼면 기체 예상 σ 가 내려가 여유가 그만큼 넓어진다. 생산 σ 는 slope_only ⟨outputs/report06_derived.json : modes.production_mode⟩ 라 기울기만 받는다."}, {"주장": "세 밴드를 한 세션에서 재어 밴드별 σ 오차를 진폭 재현성으로 묶는다.", "근거": "그림 5 · outputs/report06_derived.json:slope.gap_db_min", "공격": "밴드별 독립 σ 오차 1 dB 에서 단일자세 순위 보존확률이 0.58 ⟨outputs/report06_derived.json : ranking_validation.p_order_preserved_at_1db.matrice4e⟩ 로 떨어진다.", "답": "그 값은 단일자세 인용의 것이다 ⟨outputs/report06_derived.json : ranking_validation.mc_basis⟩. 캠페인은 방위 1.38° ⟨outputs/report06_derived.json : ranking_validation.az_step_deg⟩ 표본으로 자세평균 σ 를 내고, 자세평균에서 다섯 기체 순위가 일치한다."}, {"주장": "원거리장 거리를 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ 로 잡고 세 D 정의를 한 표에 함께 싣는다.", "근거": "그림 2 · outputs/report06_derived.json:farfield_adopted.R_ff_max_m", "공격": "D 정의를 모터대각으로 바꾸면 요구거리가 3.65 배 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩ 달라진다.", "답": "그 비를 표에 실었고 채택은 가장 보수적인 env 다 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩."}, {"주장": "두 기체를 한 캠페인에서 재어 크기전이 법칙을 부호 하나로 고른다.", "근거": "그림 6 · outputs/report06_derived.json:size_law.differential_db", "공격": "기체 2종은 표본이 작다.", "답": "두 기체가 앵커보다 각각 크고 작아 L² 와 L⁴ 가 반대 부호를 예측하고, 차등은 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 다."}, {"주장": "Pfa 교정은 통제 몬테카를로가 세우고, 야외 세션은 부지 경험 Pfa 를 별도 값으로 기록한다.", "근거": "§4 결정표 · outputs/verify_cfar.json:meta.runtime_s", "공격": "야외에서 Pfa 를 통제하면 교정 주장이 더 강해진다.", "답": "CFAR 임계는 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ 의 GPU 몬테카를로 경험 Pfa 로 교정했다. 야외 세션은 같은 임계를 부지 잡음 위에서 재현해 사슬을 확인한다 ⟨outputs/verify_cfar.json : chain_verify⟩."}, {"주장": "12-bit ADC 동적범위 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ 가 직접파 제거의 천장이다.", "근거": "그림 1 · outputs/report06_measurement.json:adc.headroom_db_min", "공격": "시뮬 자유공간 기하의 DNR 은 야외보다 낙관적이다.", "답": "가장 좁은 여유는 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ 이고, 세션은 직접파를 실제로 받아 그 여유를 측정값으로 대체한다."}]}-->
## §4.5 방어선

| 주장 | 근거 | 공격 | 답 |
|---|---|---|---|
| 야외 캠페인은 검출 사슬과 세 파형의 상대순위를 결판낸다. | §3 · §4 결정표 · outputs/sigma_sensitivity.json:aspect_averaged.consensus_order | 야외 환경이 시뮬 자유공간과 달라 비교 대상이 흐려진다. | 순위를 정하는 λ²·점유·대역폭 항은 밴드 간 차이고 환경 항은 세 밴드 공통이다. 자세평균 σ 에서 기체 5 종 ⟨outputs/report06_derived.json : ranking_validation.n_drones⟩이 같은 순위에 합의한다 ⟨outputs/sigma_sensitivity.json : aspect_averaged.all_drones_agree⟩. |
| 세션 드리프트 예산 1.00 dB ⟨outputs/report06_derived.json : ranking_validation.drift_budget_db⟩ 가 자세평균 뒤집힘 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 아래에 있다. | §3 · outputs/report06_derived.json:ranking_validation.drift_margin_db | 여유 0.30 dB ⟨outputs/report06_derived.json : ranking_validation.drift_margin_db⟩ 는 얇다. | 좁은 쪽은 Matrice 4E 하나이고 Mini 5 Pro 는 2.95 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_db.mini5pro⟩ 다. 두 기체를 함께 재어 넓은 쪽이 좁은 쪽의 판정을 받쳐 준다. |
| 교정구를 표적 자리에서 세션 시작·끝에 재어 절대 레벨의 첫 측정 앵커를 세운다. | 그림 3 · outputs/report06_derived.json:calibration_margin_min_db | 그 여유는 앵커 절대레벨 위의 설계값이고, 앵커 절편에는 통계 규약 변환 상수가 들어 있다. | 그 상수는 2.51 dB ⟨outputs/sigma_anchor.json : statistic_resolution.reconcile.by_kind.exponential.offset_db⟩ 이고, 빼면 기체 예상 σ 가 내려가 여유가 그만큼 넓어진다. 생산 σ 는 slope_only ⟨outputs/report06_derived.json : modes.production_mode⟩ 라 기울기만 받는다. |
| 세 밴드를 한 세션에서 재어 밴드별 σ 오차를 진폭 재현성으로 묶는다. | 그림 5 · outputs/report06_derived.json:slope.gap_db_min | 밴드별 독립 σ 오차 1 dB 에서 단일자세 순위 보존확률이 0.58 ⟨outputs/report06_derived.json : ranking_validation.p_order_preserved_at_1db.matrice4e⟩ 로 떨어진다. | 그 값은 단일자세 인용의 것이다 ⟨outputs/report06_derived.json : ranking_validation.mc_basis⟩. 캠페인은 방위 1.38° ⟨outputs/report06_derived.json : ranking_validation.az_step_deg⟩ 표본으로 자세평균 σ 를 내고, 자세평균에서 다섯 기체 순위가 일치한다. |
| 원거리장 거리를 24.44 m ⟨outputs/report06_derived.json : farfield_adopted.R_ff_max_m⟩ 로 잡고 세 D 정의를 한 표에 함께 싣는다. | 그림 2 · outputs/report06_derived.json:farfield_adopted.R_ff_max_m | D 정의를 모터대각으로 바꾸면 요구거리가 3.65 배 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩ 달라진다. | 그 비를 표에 실었고 채택은 가장 보수적인 env 다 ⟨outputs/report06_derived.json : farfield_adopted.spread_ratio_max⟩. |
| 두 기체를 한 캠페인에서 재어 크기전이 법칙을 부호 하나로 고른다. | 그림 6 · outputs/report06_derived.json:size_law.differential_db | 기체 2종은 표본이 작다. | 두 기체가 앵커보다 각각 크고 작아 L² 와 L⁴ 가 반대 부호를 예측하고, 차등은 4.06 dB ⟨outputs/report06_derived.json : size_law.differential_db⟩ 다. |
| Pfa 교정은 통제 몬테카를로가 세우고, 야외 세션은 부지 경험 Pfa 를 별도 값으로 기록한다. | §4 결정표 · outputs/verify_cfar.json:meta.runtime_s | 야외에서 Pfa 를 통제하면 교정 주장이 더 강해진다. | CFAR 임계는 2717 s ⟨outputs/verify_cfar.json : meta.runtime_s⟩ 의 GPU 몬테카를로 경험 Pfa 로 교정했다. 야외 세션은 같은 임계를 부지 잡음 위에서 재현해 사슬을 확인한다 ⟨outputs/verify_cfar.json : chain_verify⟩. |
| 12-bit ADC 동적범위 74.01 dB ⟨outputs/report06_measurement.json : hw.dynamic_range_db⟩ 가 직접파 제거의 천장이다. | 그림 1 · outputs/report06_measurement.json:adc.headroom_db_min | 시뮬 자유공간 기하의 DNR 은 야외보다 낙관적이다. | 가장 좁은 여유는 19.2 dB ⟨outputs/report06_measurement.json : adc.headroom_db_min⟩ 이고, 세션은 직접파를 실제로 받아 그 여유를 측정값으로 대체한다. |

### §4.5 인용

1. Das et al., "Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling", IEEE Wireless Communications Letters 15:3731-3735, 2026 [게재] doi:10.1109/LWC.2026.3705634 (Phantom 3 · Table III — 주파수 기울기 앵커(§4-1))<!--pk:cite {"kind": "cite", "authors": "Das et al.", "title": "Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling", "venue": "IEEE Wireless Communications Letters", "volume": "15", "pages": "3731-3735", "year": 2026, "status": "published", "status_ko": "게재", "arxiv": null, "doi": "10.1109/LWC.2026.3705634", "note": "Phantom 3 · Table III — 주파수 기울기 앵커(§4-1)", "text": "Das et al., \"Multiband Monostatic and Bistatic RCS Characterization of AAVs for ISAC Channel Modeling\", IEEE Wireless Communications Letters 15:3731-3735, 2026 [게재] doi:10.1109/LWC.2026.3705634 (Phantom 3 · Table III — 주파수 기울기 앵커(§4-1))"}-->
2. Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (FDTD RCS → 패시브 커버리지 → 50 m OTA 검출. 같은 산출물의 게재 전례)<!--pk:cite {"kind": "cite", "authors": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski", "title": "Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band", "venue": "NATO STO-MP-MSG-SET-183, paper 13", "volume": null, "pages": null, "year": 2021, "status": "published", "status_ko": "게재", "arxiv": null, "doi": null, "note": "FDTD RCS → 패시브 커버리지 → 50 m OTA 검출. 같은 산출물의 게재 전례", "text": "Rzewuski, Kulpa, Pachwicewicz, Malanowski, Salski, \"Drone Detectability Feasibility Study using Passive Radars Operating in WIFI and DVB-T Band\", NATO STO-MP-MSG-SET-183, paper 13, 2021 [게재] (FDTD RCS → 패시브 커버리지 → 50 m OTA 검출. 같은 산출물의 게재 전례)"}-->
3. NI / Ettus Research, "USRP X410 Specifications", National Instruments technical specification, 2024 [기술보고서] (`src/experiment_x410.py:61` 이 인용한 원사양 — 4 TX/4 RX · 400 MHz · 12-bit)<!--pk:cite {"kind": "cite", "authors": "NI / Ettus Research", "title": "USRP X410 Specifications", "venue": "National Instruments technical specification", "volume": null, "pages": null, "year": 2024, "status": "tech report", "status_ko": "기술보고서", "arxiv": null, "doi": null, "note": "`src/experiment_x410.py:61` 이 인용한 원사양 — 4 TX/4 RX · 400 MHz · 12-bit", "text": "NI / Ettus Research, \"USRP X410 Specifications\", National Instruments technical specification, 2024 [기술보고서] (`src/experiment_x410.py:61` 이 인용한 원사양 — 4 TX/4 RX · 400 MHz · 12-bit)"}-->

In [ ]:
# 이 편의 숫자를 직접 열어보기 — 표·그림의 모든 값은 아래 JSON 에서 나온다.
import json
D = json.load(open('outputs/report06_derived.json'))
print(json.dumps(D['_meta']['definitions'], ensure_ascii=False, indent=1))
print('원거리장 채택 :', D['farfield_adopted'])
print('기울기 판별폭 :', [(r['airframe'], round(r['gap_db'], 2))
                          for r in D['slope']['rows']])
print('생산 모드     :', D['modes']['production_mode'],
      '· 평균 레벨이동', round(D['modes']['level_shift_production_abs_max_db'], 2), 'dB')
print('크기법칙 차등 :', round(D['size_law']['differential_db'], 2), 'dB')
print('순위 판정     :', D['ranking_validation']['consensus_order'],
      '· 뒤집힘 폭', round(D['ranking_validation']['flip_span_min_db'], 2),
      'dB · 드리프트 여유', round(D['ranking_validation']['drift_margin_db'], 2), 'dB')

## §5. 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 야외 고정기하에서 세 파형을 같은 세션에 송신하고 방위 스윕으로 자세평균 순서를 잰다 | 파형 상대순위 `L1 > G1 > W1` 가 실측에서 확인된다 — 판정 문턱은 뒤집힘 폭 1.30 dB ⟨outputs/report06_derived.json : ranking_validation.flip_span_min_db⟩ 다 | 06편 §3 · §4 → 05편 결과와 대조 |
| 교정구를 표적과 같은 자리·같은 높이에서 세션 시작과 끝에 잰다 | 지금 우리 PO 출력인 절대 레벨이 처음으로 측정에 앵커된다 — 생산 모드의 평균 레벨이동 0.00 dB ⟨outputs/report06_derived.json : modes.level_shift_production_abs_max_db⟩ 가 측정값으로 대체된다 | 06편 §2-2 · §3-2 → `src/sigma_anchor.py` 레벨 앵커 등록 |
| 기체 2종을 입고하고 §2 체크리스트 6항목대로 세션을 돌린다 | 우리 기체의 절대 σ(f, φ) 가 외부 앵커 없이 자체 측정으로 선다 | `outputs/measured_sigma.json` → `src/sigma_anchor.py:255` 앵커 등록 |
| 세 밴드를 같은 세션에서 재고 세션간 진폭 재현성을 기록한다 | 밴드 기울기가 우리 커널 값과 앵커 0.210 ⟨outputs/report06_derived.json : slope.anchor_db_per_ghz⟩ dB/GHz 중 어디에 앉는지 결정된다 | 06편 §4-1 → 02편 §4 재기술 |
| 두 기체를 한 캠페인에서 재고 μ 차이의 부호를 본다 | 크기전이 법칙 L² vs L⁴ (원장 9.50 dB ⟨outputs/report06_derived.json : size_law.uncontrolled_size_db⟩)가 확정된다 | 06편 §4-2 → `src/sigma_anchor.py` 크기법칙 고정 |
| VV / VH / HV / HH 4조합을 잰다 | 무편파 스칼라 모형과 VV 측정의 차가 dB 로 확정된다 | `src/materials.py:171` `gamma_po()` 의 편파 확장 결정 |
| β ≤ 45° 안에서 송수신 분리각별 기하를 §2-1·§2-5 방식으로 계산한다 | 바이스태틱 세션의 원거리장 거리와 게이팅 임계가 정해진다 | `benchmark/plan_measurement.py` 확장 |
| 같은 세션에서 모서리가 많은 표준체(평판·이면각)를 함께 잰다 | 1차 PTD 항의 부호와 크기를 실측이 심판한다 — 평판 RMS 시험은 위상맹목이고, 켠 비용은 +47.2% ⟨outputs/ptd_wiring.json : verdict.cost_increase_pct⟩ 다 | 06편 §2-2 확장 · §4 결정표 → `benchmark/ptd_plate_validation.py` |
| 자세축과 로터위상축을 σ 생산자에 배선한 뒤 sim-to-sim ablation 을 돌린다 | 우리 σ 가 (σ̄, τ_decorr, 분포형) 3개 수로 환원되는지가 실측 없이 판정된다 — 태스크는 분류가 아니라 **검출**로 고정한다(통계 RCS 와 큐브에는 로터가 없어 분류는 로터 유무를 재는 실험이 된다) | `docs/SIM2REAL_PLAN.md` → `outputs/s2r_protocol.json` |
| 3팔 실측 ablation 설계를 검출 태스크와 supervision 사다리로 다시 짠다 | 현 설계 판정 RISKY ⟨outputs/s2r_attack.json : verdict⟩ 의 근거가 닫힌다 — 자세축·로터위상축을 배선하기 전에는 세 팔이 분포형·상관시간 두 수로 환원되어 판정이 설계로 보장된다 | `outputs/s2r_attack.json` → `docs/SIM2REAL_PLAN.md` |
| 정지 로터 세션과 별도로 회전 세션을 잡는다 | 마이크로도플러가 앵커와의 사과-대-사과를 깨지 않고 들어온다 | 06편 §2-4 → future work |